# Market Agent

LLM 판단과 보수적 Timing Readiness를 포함한 최종 코드입니다.

API 키는 `./../OpenAI_key.txt`에서 읽습니다.

In [1]:

# ============================================================
# Market Agent with LLM Judgment + Conservative Timing Readiness
# ============================================================
# 필요 패키지:
# pip install openai finance-datareader yfinance requests lxml html5lib beautifulsoup4

from datetime import datetime
from pathlib import Path
from io import StringIO
import time
import re
import json
import os
import sys

import numpy as np
import pandas as pd
import yfinance as yf
import FinanceDataReader as fdr
import requests
from openai import OpenAI


# ============================================================
# [1] 기본 설정
# ============================================================

START_DATE = "2020-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

OUT_DIR = Path("./market_agent_data")
KEY_FILE = Path("./../OpenAI_key.txt")

PRICE_FILE = OUT_DIR / "market_price_data.csv"
FLOW_FILE = OUT_DIR / "market_flow_data.csv"
FEATURE_FILE = OUT_DIR / "market_feature_data.csv"
REPORT_JSON_FILE = OUT_DIR / "market_agent_report.json"
REPORT_TABLE_FILE = OUT_DIR / "market_agent_report_table.csv"
FINAL_TXT_FILE = OUT_DIR / "market_agent_final_report.txt"
LLM_REPORT_FILE = OUT_DIR / "market_agent_llm_report.txt"
LLM_JSON_FILE = OUT_DIR / "market_agent_llm_report.json"
INTEGRATION_PAYLOAD_FILE = OUT_DIR / "market_agent_integration_payload.json"

KR_STOCKS = {
    "005930": "Samsung Electronics",
    "000660": "SK Hynix",
    "042700": "Hanmi Semiconductor",
}

KR_INDEXES = {
    "KS11": "KOSPI",
    "KQ11": "KOSDAQ",
}

GLOBAL_ASSETS = {
    "SOXX": ("iShares Semiconductor ETF", "US_ETF"),
    "SMH": ("VanEck Semiconductor ETF", "US_ETF"),
    "^IXIC": ("NASDAQ Composite", "US_INDEX"),
    "^GSPC": ("S&P 500", "US_INDEX"),
    "KRW=X": ("USD/KRW", "FX"),
}


def ensure_output_dir():
    OUT_DIR.mkdir(parents=True, exist_ok=True)


def read_openai_key(key_file=KEY_FILE):
    key_file = Path(key_file)
    print("[INFO] Current working directory:", Path.cwd())
    print("[INFO] OpenAI key file path:", key_file.resolve())

    if not key_file.exists():
        raise FileNotFoundError(
            f"OpenAI API key file not found: {key_file.resolve()}\n"
            "OpenAI_key.txt 파일을 현재 노트북 위치의 상위 폴더에 두세요."
        )

    key = key_file.read_text(encoding="utf-8").strip()

    # OpenAI_key.txt 안에 OPENAI_API_KEY=sk-... 형태로 적은 경우도 처리
    if "=" in key and "OPENAI_API_KEY" in key:
        key = key.split("=", 1)[1].strip()

    key = key.strip().strip('"').strip("'")

    if not key.startswith("sk-"):
        raise ValueError(
            "OpenAI API key format is invalid. "
            "OpenAI_key.txt에는 sk-로 시작하는 키만 넣으세요."
        )

    return key


# ============================================================
# [2] Utility 함수
# ============================================================

def clean_number(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(",", "")
    x = x.replace("%", "")
    x = x.replace("+", "")
    x = x.replace("−", "-")
    x = x.replace("▲", "")
    x = x.replace("▼", "-")
    x = re.sub(r"[^0-9.\-]", "", x)

    if x in ["", "-", "."]:
        return np.nan

    try:
        return float(x)
    except Exception:
        return np.nan


def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x).strip() for x in col if str(x) != "nan"])
            for col in df.columns
        ]
    else:
        df.columns = [str(c).strip() for c in df.columns]

    return df


def find_col(columns, must_include=None, exclude=None):
    must_include = must_include or []
    exclude = exclude or []

    for col in columns:
        col_str = str(col).strip()
        if all(key in col_str for key in must_include) and not any(key in col_str for key in exclude):
            return col

    return None


def safe_value(row, col):
    val = row.get(col, np.nan)
    if pd.isna(val):
        return None
    return val


def fmt_pct(x):
    if x is None or pd.isna(x):
        return "N/A"
    return f"{x * 100:.2f}%"


def fmt_num(x):
    if x is None or pd.isna(x):
        return "N/A"
    return f"{x:,.0f}"


def to_long_price_df(df, asset_code, asset_name, market, source):
    columns = [
        "Date", "AssetCode", "AssetName", "Market", "Source",
        "Open", "High", "Low", "Close", "Volume", "Value"
    ]

    if df is None or df.empty:
        return pd.DataFrame(columns=columns)

    out = df.copy().reset_index()

    if "Date" not in out.columns:
        out = out.rename(columns={out.columns[0]: "Date"})

    for col in ["Open", "High", "Low", "Close", "Volume"]:
        if col not in out.columns:
            out[col] = np.nan

    if "Value" not in out.columns:
        out["Value"] = np.nan

    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["AssetCode"] = asset_code
    out["AssetName"] = asset_name
    out["Market"] = market
    out["Source"] = source

    out = out[columns].copy()
    out = out.dropna(subset=["Date"])
    out = out.sort_values("Date").reset_index(drop=True)

    return out


def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    return rsi


def timing_zone(score):
    score = int(max(0, min(100, score)))

    if score >= 80:
        return "Strong Entry Zone"
    if score >= 60:
        return "Entry Candidate"
    if score >= 40:
        return "Neutral"
    if score >= 20:
        return "Defensive"
    return "Avoid"


# ============================================================
# [3] Data Collector
# ============================================================

class MarketDataCollector:
    def __init__(self, start_date, end_date, kr_stocks, kr_indexes, global_assets):
        self.start_date = start_date
        self.end_date = end_date
        self.kr_stocks = kr_stocks
        self.kr_indexes = kr_indexes
        self.global_assets = global_assets

    def fetch_fdr_kr_stock_prices(self):
        frames = []

        for ticker, asset_name in self.kr_stocks.items():
            try:
                raw = fdr.DataReader(ticker, self.start_date, self.end_date)

                price_df = to_long_price_df(
                    df=raw,
                    asset_code=ticker,
                    asset_name=asset_name,
                    market="KR_STOCK",
                    source="FinanceDataReader"
                )

                if not price_df.empty:
                    frames.append(price_df)
                    print(f"[OK] FDR stock: {ticker}, rows={len(price_df)}")
                else:
                    print(f"[WARN] FDR stock empty: {ticker}")

            except Exception as e:
                print(f"[ERROR] FDR stock failed: {ticker} / {e}")

            time.sleep(0.2)

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def fetch_fdr_kr_index_prices(self):
        frames = []

        for code, name in self.kr_indexes.items():
            try:
                raw = fdr.DataReader(code, self.start_date, self.end_date)

                price_df = to_long_price_df(
                    df=raw,
                    asset_code=code,
                    asset_name=name,
                    market="KR_INDEX",
                    source="FinanceDataReader"
                )

                if not price_df.empty:
                    frames.append(price_df)
                    print(f"[OK] FDR index: {code}, rows={len(price_df)}")
                else:
                    print(f"[WARN] FDR index empty: {code}")

            except Exception as e:
                print(f"[ERROR] FDR index failed: {code} / {e}")

            time.sleep(0.2)

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def fetch_yfinance_global_prices(self):
        frames = []

        for ticker, (asset_name, market) in self.global_assets.items():
            try:
                raw = yf.download(
                    ticker,
                    start=self.start_date,
                    end=self.end_date,
                    auto_adjust=False,
                    progress=False
                )

                if raw is None or raw.empty:
                    print(f"[WARN] yfinance empty: {ticker}")
                    continue

                if isinstance(raw.columns, pd.MultiIndex):
                    raw.columns = raw.columns.get_level_values(0)

                keep_cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in raw.columns]
                raw = raw[keep_cols].copy()

                price_df = to_long_price_df(
                    df=raw,
                    asset_code=ticker,
                    asset_name=asset_name,
                    market=market,
                    source="yfinance"
                )

                if not price_df.empty:
                    frames.append(price_df)
                    print(f"[OK] yfinance: {ticker}, rows={len(price_df)}")
                else:
                    print(f"[WARN] yfinance converted empty: {ticker}")

            except Exception as e:
                print(f"[ERROR] yfinance failed: {ticker} / {e}")

            time.sleep(0.2)

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def fetch_naver_flow_one_stock(self, ticker, asset_name, max_pages=250, debug_first_page=False):
        columns = [
            "Date", "AssetCode", "AssetName", "Market", "Source",
            "InvestorType", "NetBuyVolume"
        ]

        frames = []
        headers = {"User-Agent": "Mozilla/5.0"}

        for page in range(1, max_pages + 1):
            url = f"https://finance.naver.com/item/frgn.naver?code={ticker}&page={page}"

            try:
                res = requests.get(url, headers=headers, timeout=10)
                res.raise_for_status()

                tables = pd.read_html(StringIO(res.text), encoding="cp949")

                target = None
                date_col = None
                foreign_col = None
                institution_col = None

                if debug_first_page and page == 1:
                    print(f"\n[DEBUG] Naver tables for {ticker}, page={page}, n_tables={len(tables)}")
                    for i, t in enumerate(tables):
                        t = flatten_columns(t.copy())
                        print(f"[DEBUG] table={i}, columns={list(t.columns)}")

                for table in tables:
                    table = flatten_columns(table.copy())
                    table = table.dropna(how="all")

                    if table.empty:
                        continue

                    cols = list(table.columns)

                    date_candidate = find_col(cols, must_include=["날짜"])
                    foreign_candidate = find_col(
                        cols,
                        must_include=["외국인"],
                        exclude=["보유", "비율", "소진"]
                    )
                    institution_candidate = find_col(
                        cols,
                        must_include=["기관"],
                        exclude=["보유", "비율", "소진"]
                    )

                    if date_candidate is not None and (
                        foreign_candidate is not None or institution_candidate is not None
                    ):
                        target = table.copy()
                        date_col = date_candidate
                        foreign_col = foreign_candidate
                        institution_col = institution_candidate
                        break

                if target is None or target.empty:
                    continue

                target["Date"] = pd.to_datetime(target[date_col], errors="coerce")
                target = target.dropna(subset=["Date"])

                if target.empty:
                    continue

                oldest_on_page = target["Date"].min()

                target = target[target["Date"] >= pd.to_datetime(self.start_date)]
                target = target[target["Date"] <= pd.to_datetime(self.end_date)]

                if target.empty:
                    if oldest_on_page < pd.to_datetime(self.start_date):
                        break
                    continue

                page_rows = []

                if foreign_col is not None:
                    temp = target[["Date", foreign_col]].copy()
                    temp["InvestorType"] = "Foreign"
                    temp["NetBuyVolume"] = temp[foreign_col].apply(clean_number)
                    page_rows.append(temp[["Date", "InvestorType", "NetBuyVolume"]])

                if institution_col is not None:
                    temp = target[["Date", institution_col]].copy()
                    temp["InvestorType"] = "Institution"
                    temp["NetBuyVolume"] = temp[institution_col].apply(clean_number)
                    page_rows.append(temp[["Date", "InvestorType", "NetBuyVolume"]])

                if page_rows:
                    page_df = pd.concat(page_rows, ignore_index=True)
                    page_df["AssetCode"] = ticker
                    page_df["AssetName"] = asset_name
                    page_df["Market"] = "KR_STOCK"
                    page_df["Source"] = "NaverFinance"
                    page_df = page_df[columns]
                    frames.append(page_df)

            except Exception as e:
                print(f"[WARN] Naver flow page failed: {ticker}, page={page}, error={e}")

            time.sleep(0.15)

        if not frames:
            print(f"[WARN] Naver flow empty: {ticker}")
            return pd.DataFrame(columns=columns)

        result = pd.concat(frames, ignore_index=True)
        result = result.dropna(subset=["Date"])
        result = result.dropna(subset=["NetBuyVolume"])
        result = result.drop_duplicates(subset=["Date", "AssetCode", "InvestorType"])
        result = result.sort_values(["Date", "AssetCode", "InvestorType"]).reset_index(drop=True)

        print(f"[OK] Naver flow: {ticker}, rows={len(result)}")
        return result

    def fetch_naver_flows(self, max_pages=250):
        columns = [
            "Date", "AssetCode", "AssetName", "Market", "Source",
            "InvestorType", "NetBuyVolume"
        ]

        frames = []

        for ticker, asset_name in self.kr_stocks.items():
            flow_df = self.fetch_naver_flow_one_stock(
                ticker=ticker,
                asset_name=asset_name,
                max_pages=max_pages,
                debug_first_page=False
            )

            if not flow_df.empty:
                frames.append(flow_df)

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)

    def collect(self, max_pages=250):
        ensure_output_dir()
        print("[START] Data collection")

        kr_stock_price_df = self.fetch_fdr_kr_stock_prices()
        kr_index_price_df = self.fetch_fdr_kr_index_prices()
        global_price_df = self.fetch_yfinance_global_prices()
        flow_df = self.fetch_naver_flows(max_pages=max_pages)

        price_frames = [df for df in [kr_stock_price_df, kr_index_price_df, global_price_df] if df is not None and not df.empty]

        if price_frames:
            price_df = pd.concat(price_frames, ignore_index=True)
        else:
            price_df = pd.DataFrame(columns=[
                "Date", "AssetCode", "AssetName", "Market", "Source",
                "Open", "High", "Low", "Close", "Volume", "Value"
            ])

        price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
        price_df = price_df.dropna(subset=["Date"])
        price_df = price_df.sort_values(["Date", "AssetCode"]).drop_duplicates().reset_index(drop=True)

        flow_df["Date"] = pd.to_datetime(flow_df["Date"], errors="coerce")
        flow_df = flow_df.dropna(subset=["Date"])
        flow_df = flow_df.sort_values(["Date", "AssetCode", "InvestorType"]).drop_duplicates().reset_index(drop=True)

        ensure_output_dir()
        price_df.to_csv(PRICE_FILE, index=False, encoding="utf-8-sig")
        flow_df.to_csv(FLOW_FILE, index=False, encoding="utf-8-sig")

        print("[DONE] Data collection")
        print("price:", PRICE_FILE, price_df.shape)
        print("flow :", FLOW_FILE, flow_df.shape)

        return price_df, flow_df


# ============================================================
# [4] Feature Builder
# ============================================================

class MarketFeatureBuilder:
    def add_price_features(self, group):
        group = group.sort_values("Date").copy()

        close = group["Close"]
        volume = group["Volume"]

        group["Return_1D"] = close.pct_change(1)
        group["Return_5D"] = close.pct_change(5)
        group["Return_20D"] = close.pct_change(20)
        group["Return_60D"] = close.pct_change(60)
        group["Return_120D"] = close.pct_change(120)
        group["Return_240D"] = close.pct_change(240)

        group["MA30"] = close.rolling(30).mean()
        group["MA60"] = close.rolling(60).mean()
        group["MA120"] = close.rolling(120).mean()

        group["MA30_Above_MA60"] = (group["MA30"] > group["MA60"]).astype(int)
        group["MA60_Above_MA120"] = (group["MA60"] > group["MA120"]).astype(int)
        group["MA_Alignment"] = (
            (group["MA30"] > group["MA60"]) &
            (group["MA60"] > group["MA120"])
        ).astype(int)

        group["Close_to_MA30"] = close / group["MA30"] - 1
        group["Close_to_MA60"] = close / group["MA60"] - 1
        group["Close_to_MA120"] = close / group["MA120"] - 1

        group["High_52W"] = close.rolling(240).max()
        group["Drawdown_52W"] = close / group["High_52W"] - 1

        group["Volatility_20D"] = group["Return_1D"].rolling(20).std()
        group["Volatility_60D"] = group["Return_1D"].rolling(60).std()

        group["RSI14"] = compute_rsi(close, window=14)

        ema12 = close.ewm(span=12, adjust=False).mean()
        ema26 = close.ewm(span=26, adjust=False).mean()

        group["MACD"] = ema12 - ema26
        group["MACD_Signal"] = group["MACD"].ewm(span=9, adjust=False).mean()
        group["MACD_Hist"] = group["MACD"] - group["MACD_Signal"]

        group["MACD_GoldenCross"] = (
            (group["MACD"] > group["MACD_Signal"]) &
            (group["MACD"].shift(1) <= group["MACD_Signal"].shift(1))
        ).astype(int)

        group["ROC60"] = close / close.shift(60) - 1

        group["Volume_MA20"] = volume.rolling(20).mean()
        group["Volume_Ratio_20D"] = volume / group["Volume_MA20"]

        return group

    def add_flow_features(self, group):
        group = group.sort_values("Date").copy()

        group["Foreign_5D"] = group["Foreign"].rolling(5).sum()
        group["Foreign_20D"] = group["Foreign"].rolling(20).sum()
        group["Institution_5D"] = group["Institution"].rolling(5).sum()
        group["Institution_20D"] = group["Institution"].rolling(20).sum()

        group["TotalFlow"] = group["Foreign"] + group["Institution"]
        group["TotalFlow_5D"] = group["TotalFlow"].rolling(5).sum()
        group["TotalFlow_20D"] = group["TotalFlow"].rolling(20).sum()

        return group

    def build_relative_strength_features(self, price_df, feature_df):
        close_wide = (
            price_df
            .pivot_table(index="Date", columns="AssetCode", values="Close", aggfunc="last")
            .sort_index()
        )

        relative_rows = []

        relative_pair_specs = [
            ("005930", "KS11", "RS_vs_KOSPI"),
            ("000660", "KS11", "RS_vs_KOSPI"),
            ("042700", "KS11", "RS_vs_KOSPI"),
            ("SOXX", "^IXIC", "RS_vs_NASDAQ"),
            ("SMH", "^IXIC", "RS_vs_NASDAQ"),
            ("SOXX", "^GSPC", "RS_vs_SP500"),
            ("SMH", "^GSPC", "RS_vs_SP500"),
        ]

        for asset, benchmark, rs_col in relative_pair_specs:
            if asset not in close_wide.columns or benchmark not in close_wide.columns:
                continue

            rs = close_wide[asset] / close_wide[benchmark]

            temp = pd.DataFrame({
                "Date": close_wide.index,
                "AssetCode": asset,
                rs_col: rs.values,
                f"{rs_col}_Return_20D": rs.pct_change(20).values,
                f"{rs_col}_Return_60D": rs.pct_change(60).values,
            })

            relative_rows.append(temp.reset_index(drop=True))

        if relative_rows:
            relative_df = pd.concat(relative_rows, axis=0, ignore_index=True)

            relative_df = (
                relative_df
                .groupby(["Date", "AssetCode"], as_index=False)
                .first()
            )

            feature_df = feature_df.merge(
                relative_df,
                on=["Date", "AssetCode"],
                how="left"
            )

        return feature_df

    def build(self, price_df=None, flow_df=None):
        ensure_output_dir()

        if price_df is None:
            price_df = pd.read_csv(PRICE_FILE)

        if flow_df is None:
            flow_df = pd.read_csv(FLOW_FILE)

        price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
        flow_df["Date"] = pd.to_datetime(flow_df["Date"], errors="coerce")

        price_df = price_df.dropna(subset=["Date"]).copy()
        flow_df = flow_df.dropna(subset=["Date"]).copy()

        price_df = price_df.sort_values(["AssetCode", "Date"]).reset_index(drop=True)
        flow_df = flow_df.sort_values(["AssetCode", "InvestorType", "Date"]).reset_index(drop=True)

        if price_df.empty:
            raise ValueError("price_df가 비어 있습니다. 데이터 수집 결과를 확인하세요.")

        price_feature_df = (
            price_df
            .groupby("AssetCode", group_keys=False)
            .apply(self.add_price_features)
            .reset_index(drop=True)
        )

        if flow_df.empty:
            flow_feature_df = pd.DataFrame(columns=[
                "Date", "AssetCode", "Foreign", "Institution",
                "Foreign_5D", "Foreign_20D",
                "Institution_5D", "Institution_20D",
                "TotalFlow", "TotalFlow_5D", "TotalFlow_20D"
            ])
        else:
            flow_wide = (
                flow_df
                .pivot_table(
                    index=["Date", "AssetCode", "AssetName", "Market", "Source"],
                    columns="InvestorType",
                    values="NetBuyVolume",
                    aggfunc="sum"
                )
                .reset_index()
            )

            flow_wide.columns.name = None

            for col in ["Foreign", "Institution"]:
                if col not in flow_wide.columns:
                    flow_wide[col] = 0.0

            flow_wide = flow_wide.sort_values(["AssetCode", "Date"]).reset_index(drop=True)

            flow_feature_df = (
                flow_wide
                .groupby("AssetCode", group_keys=False)
                .apply(self.add_flow_features)
                .reset_index(drop=True)
            )

        flow_cols = [
            "Date", "AssetCode",
            "Foreign", "Institution",
            "Foreign_5D", "Foreign_20D",
            "Institution_5D", "Institution_20D",
            "TotalFlow", "TotalFlow_5D", "TotalFlow_20D"
        ]

        feature_df = price_feature_df.merge(
            flow_feature_df[flow_cols],
            on=["Date", "AssetCode"],
            how="left"
        )

        feature_df = self.build_relative_strength_features(price_df, feature_df)

        feature_df = feature_df.sort_values(["Date", "AssetCode"]).reset_index(drop=True)

        ensure_output_dir()
        feature_df.to_csv(FEATURE_FILE, index=False, encoding="utf-8-sig")

        print("[DONE] Feature building")
        print("feature:", FEATURE_FILE, feature_df.shape)

        return feature_df


# ============================================================
# [5] Rule-based Reporter + Timing Readiness
# ============================================================

class MarketReporter:
    def judge_trend(self, row):
        evidence = []
        limitations = []

        ma_alignment = safe_value(row, "MA_Alignment")
        close_to_ma60 = safe_value(row, "Close_to_MA60")
        close_to_ma120 = safe_value(row, "Close_to_MA120")
        return_60d = safe_value(row, "Return_60D")
        return_120d = safe_value(row, "Return_120D")
        drawdown = safe_value(row, "Drawdown_52W")

        bullish = 0
        bearish = 0

        if ma_alignment == 1:
            bullish += 1
            evidence.append("MA30 > MA60 > MA120 정배열이 형성되어 있습니다.")
        elif ma_alignment == 0:
            bearish += 1
            evidence.append("이동평균선 정배열이 형성되어 있지 않습니다.")

        if close_to_ma60 is not None:
            if close_to_ma60 > 0:
                bullish += 1
                evidence.append(f"현재 가격이 MA60 대비 {fmt_pct(close_to_ma60)} 위에 있습니다.")
            else:
                bearish += 1
                evidence.append(f"현재 가격이 MA60 대비 {fmt_pct(close_to_ma60)} 아래에 있습니다.")

        if close_to_ma120 is not None:
            if close_to_ma120 > 0:
                bullish += 1
                evidence.append(f"현재 가격이 MA120 대비 {fmt_pct(close_to_ma120)} 위에 있습니다.")
            else:
                bearish += 1
                evidence.append(f"현재 가격이 MA120 대비 {fmt_pct(close_to_ma120)} 아래에 있습니다.")

        if return_60d is not None:
            if return_60d > 0:
                bullish += 1
                evidence.append(f"최근 60거래일 수익률이 {fmt_pct(return_60d)}로 양수입니다.")
            else:
                bearish += 1
                evidence.append(f"최근 60거래일 수익률이 {fmt_pct(return_60d)}로 음수입니다.")

        if return_120d is not None:
            if return_120d > 0:
                bullish += 1
                evidence.append(f"최근 120거래일 수익률이 {fmt_pct(return_120d)}로 양수입니다.")
            else:
                bearish += 1
                evidence.append(f"최근 120거래일 수익률이 {fmt_pct(return_120d)}로 음수입니다.")

        if drawdown is not None and drawdown < -0.25:
            bearish += 1
            evidence.append(f"52주 고점 대비 낙폭이 {fmt_pct(drawdown)}로 큽니다.")

        if bullish >= 5 and bearish == 0:
            view = "Bullish"
        elif bullish >= 3 and bearish <= 1:
            view = "MildBullish"
        elif bearish >= 4:
            view = "Bearish"
        elif bearish >= 2:
            view = "MildBearish"
        else:
            view = "Neutral"

        return {
            "view": view,
            "bullish_evidence_count": bullish,
            "bearish_evidence_count": bearish,
            "evidence": evidence,
            "limitations": limitations
        }

    def judge_momentum(self, row):
        evidence = []
        risk_flags = []
        limitations = []

        rsi = safe_value(row, "RSI14")
        macd_hist = safe_value(row, "MACD_Hist")
        macd_signal = safe_value(row, "MACD_Signal")
        macd = safe_value(row, "MACD")
        roc60 = safe_value(row, "ROC60")

        positive = 0
        negative = 0

        if rsi is not None:
            if 45 <= rsi <= 65:
                positive += 1
                evidence.append(f"RSI14가 {rsi:.2f}로 안정적인 모멘텀 구간입니다.")
            elif 65 < rsi <= 75:
                positive += 1
                evidence.append(f"RSI14가 {rsi:.2f}로 강한 모멘텀을 보입니다.")
                risk_flags.append("RSI가 단기 과열 구간에 접근했습니다.")
            elif rsi > 75:
                negative += 1
                risk_flags.append(f"RSI14가 {rsi:.2f}로 과열 구간입니다.")
            elif rsi < 30:
                negative += 1
                risk_flags.append(f"RSI14가 {rsi:.2f}로 과매도 구간입니다.")
            else:
                limitations.append(f"RSI14가 {rsi:.2f}로 뚜렷한 방향성을 보이지 않습니다.")

        if macd_hist is not None:
            if macd_hist > 0:
                positive += 1
                evidence.append("MACD histogram이 양수로 단기 모멘텀이 개선되어 있습니다.")
            else:
                negative += 1
                evidence.append("MACD histogram이 음수로 단기 모멘텀이 약합니다.")

        if macd is not None and macd_signal is not None:
            if macd > macd_signal:
                positive += 1
                evidence.append("MACD가 signal line 위에 있습니다.")
            else:
                negative += 1
                evidence.append("MACD가 signal line 아래에 있습니다.")

        if roc60 is not None:
            if roc60 > 0:
                positive += 1
                evidence.append(f"ROC60이 {fmt_pct(roc60)}로 중기 모멘텀이 양호합니다.")
            else:
                negative += 1
                evidence.append(f"ROC60이 {fmt_pct(roc60)}로 중기 모멘텀이 약합니다.")

        if positive >= 3 and negative <= 1:
            view = "Positive"
        elif negative >= 3:
            view = "Negative"
        else:
            view = "Mixed"

        return {
            "view": view,
            "positive_evidence_count": positive,
            "negative_evidence_count": negative,
            "evidence": evidence,
            "risk_flags": risk_flags,
            "limitations": limitations
        }

    def judge_flow(self, row):
        evidence = []
        limitations = []

        foreign_20d = safe_value(row, "Foreign_20D")
        institution_20d = safe_value(row, "Institution_20D")
        total_flow_20d = safe_value(row, "TotalFlow_20D")

        if foreign_20d is None and institution_20d is None and total_flow_20d is None:
            return {
                "view": "NotAvailable",
                "foreign_20d": None,
                "institution_20d": None,
                "total_flow_20d": None,
                "evidence": [],
                "risk_flags": [],
                "limitations": ["해당 자산에는 외국인/기관 수급 데이터가 없습니다."]
            }

        if foreign_20d is not None:
            if foreign_20d > 0:
                evidence.append(f"외국인 20일 누적 순매수 수량이 {fmt_num(foreign_20d)}주로 양수입니다.")
            else:
                evidence.append(f"외국인 20일 누적 순매수 수량이 {fmt_num(foreign_20d)}주로 음수입니다.")

        if institution_20d is not None:
            if institution_20d > 0:
                evidence.append(f"기관 20일 누적 순매수 수량이 {fmt_num(institution_20d)}주로 양수입니다.")
            else:
                evidence.append(f"기관 20일 누적 순매수 수량이 {fmt_num(institution_20d)}주로 음수입니다.")

        if total_flow_20d is not None:
            if total_flow_20d > 0:
                evidence.append(f"외국인+기관 합산 20일 누적 수급이 {fmt_num(total_flow_20d)}주로 양수입니다.")
            else:
                evidence.append(f"외국인+기관 합산 20일 누적 수급이 {fmt_num(total_flow_20d)}주로 음수입니다.")

        foreign_pos = foreign_20d is not None and foreign_20d > 0
        foreign_neg = foreign_20d is not None and foreign_20d < 0
        inst_pos = institution_20d is not None and institution_20d > 0
        inst_neg = institution_20d is not None and institution_20d < 0
        total_pos = total_flow_20d is not None and total_flow_20d > 0
        total_neg = total_flow_20d is not None and total_flow_20d < 0

        if foreign_pos and inst_pos and total_pos:
            view = "StrongAccumulation"
        elif foreign_pos and total_pos:
            view = "Accumulation"
        elif foreign_neg and inst_pos and total_pos:
            view = "MixedAccumulation"
            limitations.append("외국인은 매도 우위이나 기관 매수와 합산 수급이 이를 상쇄하고 있습니다.")
        elif foreign_neg and inst_neg and total_neg:
            view = "StrongDistribution"
        elif foreign_neg and total_neg:
            view = "Distribution"
        elif foreign_pos and inst_neg and total_neg:
            view = "MixedDistribution"
            limitations.append("외국인은 매수 우위이나 기관 매도가 합산 수급을 약화시키고 있습니다.")
        else:
            view = "Mixed"
            limitations.append("외국인과 기관 수급 방향이 명확히 일치하지 않습니다.")

        return {
            "view": view,
            "foreign_20d": foreign_20d,
            "institution_20d": institution_20d,
            "total_flow_20d": total_flow_20d,
            "evidence": evidence,
            "risk_flags": [],
            "limitations": limitations
        }

    def judge_relative_strength(self, row):
        evidence = []
        limitations = []

        rs_kospi = safe_value(row, "RS_vs_KOSPI_Return_20D")
        rs_nasdaq = safe_value(row, "RS_vs_NASDAQ_Return_20D")
        rs_sp500 = safe_value(row, "RS_vs_SP500_Return_20D")

        values = []

        if rs_kospi is not None:
            values.append(rs_kospi)
            if rs_kospi > 0:
                evidence.append(f"KOSPI 대비 20일 상대강도 변화율이 {fmt_pct(rs_kospi)}로 개선되었습니다.")
            else:
                evidence.append(f"KOSPI 대비 20일 상대강도 변화율이 {fmt_pct(rs_kospi)}로 약화되었습니다.")

        if rs_nasdaq is not None:
            values.append(rs_nasdaq)
            if rs_nasdaq > 0:
                evidence.append(f"NASDAQ 대비 20일 상대강도 변화율이 {fmt_pct(rs_nasdaq)}로 개선되었습니다.")
            else:
                evidence.append(f"NASDAQ 대비 20일 상대강도 변화율이 {fmt_pct(rs_nasdaq)}로 약화되었습니다.")

        if rs_sp500 is not None:
            values.append(rs_sp500)
            if rs_sp500 > 0:
                evidence.append(f"S&P500 대비 20일 상대강도 변화율이 {fmt_pct(rs_sp500)}로 개선되었습니다.")
            else:
                evidence.append(f"S&P500 대비 20일 상대강도 변화율이 {fmt_pct(rs_sp500)}로 약화되었습니다.")

        if len(values) == 0:
            return {
                "view": "NotAvailable",
                "average_relative_strength_20d": None,
                "evidence": [],
                "risk_flags": [],
                "limitations": ["비교 가능한 상대강도 benchmark가 없습니다."]
            }

        avg_rs = float(np.mean(values))

        if avg_rs > 0.03:
            view = "Outperforming"
        elif avg_rs < -0.03:
            view = "Underperforming"
        else:
            view = "Neutral"

        return {
            "view": view,
            "average_relative_strength_20d": avg_rs,
            "evidence": evidence,
            "risk_flags": [],
            "limitations": limitations
        }

    def judge_risk(self, row):
        evidence = []
        risk_flags = []
        limitations = []

        drawdown = safe_value(row, "Drawdown_52W")
        vol20 = safe_value(row, "Volatility_20D")
        rsi = safe_value(row, "RSI14")

        high_risk = 0
        moderate_risk = 0

        if drawdown is not None:
            if drawdown < -0.30:
                high_risk += 1
                risk_flags.append(f"52주 고점 대비 낙폭이 {fmt_pct(drawdown)}로 큽니다.")
            elif drawdown < -0.15:
                moderate_risk += 1
                risk_flags.append(f"52주 고점 대비 낙폭이 {fmt_pct(drawdown)}로 조정 구간입니다.")
            else:
                evidence.append(f"52주 고점 대비 낙폭이 {fmt_pct(drawdown)}로 제한적입니다.")

        if vol20 is not None:
            if vol20 > 0.035:
                high_risk += 1
                risk_flags.append(f"20일 변동성이 {fmt_pct(vol20)}로 높습니다.")
            elif vol20 > 0.02:
                moderate_risk += 1
                risk_flags.append(f"20일 변동성이 {fmt_pct(vol20)}로 중간 수준입니다.")
            else:
                evidence.append(f"20일 변동성이 {fmt_pct(vol20)}로 낮은 편입니다.")

        if rsi is not None:
            if rsi > 75:
                high_risk += 1
                risk_flags.append(f"RSI14가 {rsi:.2f}로 과열 리스크가 있습니다.")
            elif rsi > 70:
                moderate_risk += 1
                risk_flags.append(f"RSI14가 {rsi:.2f}로 단기 과열에 접근했습니다.")
            elif rsi < 30:
                moderate_risk += 1
                risk_flags.append(f"RSI14가 {rsi:.2f}로 과매도 상태입니다.")

        if high_risk >= 2:
            view = "High"
        elif high_risk == 1 or moderate_risk >= 2:
            view = "Moderate"
        else:
            view = "Low"

        return {
            "view": view,
            "high_risk_count": high_risk,
            "moderate_risk_count": moderate_risk,
            "evidence": evidence,
            "risk_flags": risk_flags,
            "limitations": limitations
        }

    def judge_timing_readiness(self, row, trend, momentum, flow, relative_strength, risk):
        score = 50
        supporting = []
        caution = []

        ma_alignment = safe_value(row, "MA_Alignment")
        macd = safe_value(row, "MACD")
        macd_signal = safe_value(row, "MACD_Signal")
        macd_hist = safe_value(row, "MACD_Hist")
        rsi = safe_value(row, "RSI14")
        roc60 = safe_value(row, "ROC60")
        volume_ratio = safe_value(row, "Volume_Ratio_20D")
        drawdown = safe_value(row, "Drawdown_52W")
        vol20 = safe_value(row, "Volatility_20D")

        foreign_20d = safe_value(row, "Foreign_20D")
        institution_20d = safe_value(row, "Institution_20D")
        total_flow_20d = safe_value(row, "TotalFlow_20D")

        rs_values = [
            safe_value(row, "RS_vs_KOSPI_Return_20D"),
            safe_value(row, "RS_vs_NASDAQ_Return_20D"),
            safe_value(row, "RS_vs_SP500_Return_20D"),
        ]
        rs_values = [x for x in rs_values if x is not None]

        if ma_alignment == 1:
            score += 12
            supporting.append("이동평균 정배열")
        else:
            score -= 5
            caution.append("이동평균 정배열 부재")

        if macd is not None and macd_signal is not None:
            if macd > macd_signal:
                score += 8
                supporting.append("MACD가 signal line 상회")
            else:
                score -= 6
                caution.append("MACD가 signal line 하회")

        if macd_hist is not None:
            if macd_hist > 0:
                score += 6
                supporting.append("MACD histogram 양수")
            else:
                score -= 5
                caution.append("MACD histogram 음수")

        if rsi is not None:
            if 45 <= rsi <= 70:
                score += 10
                supporting.append("RSI가 안정적 상승 구간")
            elif 70 < rsi <= 80:
                score += 2
                caution.append("RSI 단기 과열 접근")
            elif rsi > 80:
                score -= 15
                caution.append("RSI 과열")
            elif rsi < 30:
                score -= 10
                caution.append("RSI 과매도")

        if roc60 is not None:
            if roc60 > 0:
                score += 8
                supporting.append("ROC60 양수")
            else:
                score -= 8
                caution.append("ROC60 음수")

        if total_flow_20d is not None:
            if total_flow_20d > 0:
                score += 8
                supporting.append("외국인+기관 20일 합산 수급 양수")
            else:
                score -= 8
                caution.append("외국인+기관 20일 합산 수급 음수")
        else:
            caution.append("수급 데이터 부재")

        if foreign_20d is not None and foreign_20d > 0:
            score += 4
            supporting.append("외국인 20일 순매수")
        elif foreign_20d is not None and foreign_20d < 0:
            score -= 4
            caution.append("외국인 20일 순매도")

        if institution_20d is not None and institution_20d > 0:
            score += 4
            supporting.append("기관 20일 순매수")
        elif institution_20d is not None and institution_20d < 0:
            score -= 4
            caution.append("기관 20일 순매도")

        if rs_values:
            avg_rs = float(np.mean(rs_values))
            if avg_rs > 0.03:
                score += 10
                supporting.append("benchmark 대비 상대강도 개선")
            elif avg_rs < -0.03:
                score -= 10
                caution.append("benchmark 대비 상대강도 약화")
        else:
            caution.append("상대강도 benchmark 부재")

        if volume_ratio is not None:
            if 1.0 <= volume_ratio <= 2.5:
                score += 4
                supporting.append("거래량이 평균 이상")
            elif volume_ratio > 3.0:
                score -= 3
                caution.append("거래량 급증에 따른 변동성 가능성")

        if drawdown is not None:
            if drawdown < -0.30:
                score -= 12
                caution.append("52주 고점 대비 낙폭 과대")
            elif drawdown < -0.15:
                score -= 5
                caution.append("52주 고점 대비 조정 구간")

        if vol20 is not None:
            if vol20 > 0.035:
                score -= 10
                caution.append("20일 변동성 높음")
            elif vol20 > 0.02:
                score -= 4
                caution.append("20일 변동성 중간 수준")
            else:
                score += 3
                supporting.append("20일 변동성 낮음")

        if risk["view"] == "High":
            score -= 15
            caution.append("risk view High")
        elif risk["view"] == "Moderate":
            score -= 5
            caution.append("risk view Moderate")

        score = int(max(0, min(100, round(score))))
        zone = timing_zone(score)

        if score >= 80:
            interpretation = "시장 및 기술 신호가 우호적이나 최종 매수 판단은 Integration Agent에서 검증해야 합니다."
        elif score >= 60:
            interpretation = "진입 후보 구간이나 추가 확인이 필요합니다."
        elif score >= 40:
            interpretation = "방향성이 혼재되어 보수적 관찰이 필요합니다."
        elif score >= 20:
            interpretation = "방어적 접근이 우선되는 구간입니다."
        else:
            interpretation = "신규 진입을 회피하는 것이 적절한 구간입니다."

        return {
            "score": score,
            "zone": zone,
            "interpretation": interpretation,
            "supporting_signals": supporting[:8],
            "caution_signals": caution[:8]
        }

    def synthesize_stance(self, trend, momentum, flow, relative_strength, risk):
        positive_blocks = []
        negative_blocks = []
        mixed_blocks = []

        if trend["view"] in ["Bullish", "MildBullish"]:
            positive_blocks.append("trend")
        elif trend["view"] in ["Bearish", "MildBearish"]:
            negative_blocks.append("trend")
        else:
            mixed_blocks.append("trend")

        if momentum["view"] == "Positive":
            positive_blocks.append("momentum")
        elif momentum["view"] == "Negative":
            negative_blocks.append("momentum")
        else:
            mixed_blocks.append("momentum")

        if flow["view"] in ["StrongAccumulation", "Accumulation", "MixedAccumulation"]:
            positive_blocks.append("flow")
        elif flow["view"] in ["StrongDistribution", "Distribution", "MixedDistribution"]:
            negative_blocks.append("flow")
        elif flow["view"] != "NotAvailable":
            mixed_blocks.append("flow")

        if relative_strength["view"] == "Outperforming":
            positive_blocks.append("relative_strength")
        elif relative_strength["view"] == "Underperforming":
            negative_blocks.append("relative_strength")
        elif relative_strength["view"] != "NotAvailable":
            mixed_blocks.append("relative_strength")

        if risk["view"] == "High":
            negative_blocks.append("risk")
        elif risk["view"] == "Moderate":
            mixed_blocks.append("risk")

        positive_n = len(positive_blocks)
        negative_n = len(negative_blocks)

        if positive_n >= 3 and negative_n <= 1 and risk["view"] != "High":
            stance = "Positive"
        elif negative_n >= 3 or (negative_n >= 2 and risk["view"] == "High"):
            stance = "Cautious"
        else:
            stance = "Neutral"

        available_blocks = positive_n + negative_n + len(mixed_blocks)
        dominant = max(positive_n, negative_n)

        if available_blocks == 0:
            confidence = "Low"
        elif dominant / available_blocks >= 0.70:
            confidence = "High"
        elif dominant / available_blocks >= 0.45:
            confidence = "Medium"
        else:
            confidence = "Low"

        return stance, confidence, {
            "positive_blocks": positive_blocks,
            "negative_blocks": negative_blocks,
            "mixed_blocks": mixed_blocks
        }

    def build_conflict_aware_rationale(self, row, stance, confidence, decision_basis, trend, momentum, flow, relative_strength, risk, timing):
        name = row["AssetName"]

        pos = decision_basis["positive_blocks"]
        neg = decision_basis["negative_blocks"]
        mix = decision_basis["mixed_blocks"]

        base = (
            f"{name}에 대한 Market Agent의 판단은 {stance}이며, 신뢰도는 {confidence}입니다. "
            f"우호 블록은 {pos}, 경계 블록은 {neg}, 혼재 블록은 {mix}입니다. "
            f"보수적 Timing Readiness는 {timing['score']}점({timing['zone']})입니다. "
        )

        conflict_parts = []

        if stance == "Positive" and risk["view"] == "High":
            conflict_parts.append(
                "다만 리스크가 High이므로 긍정 판단은 공격적 매수보다는 리스크 관리가 필요한 신호로 해석해야 합니다."
            )

        if trend["view"] in ["Bullish", "MildBullish"] and flow["view"] in ["Distribution", "StrongDistribution", "MixedDistribution"]:
            conflict_parts.append(
                "추세는 우호적이지만 수급이 약해 상승 지속성에 대한 검증이 필요합니다."
            )

        if trend["view"] in ["Bullish", "MildBullish"] and relative_strength["view"] == "Underperforming":
            conflict_parts.append(
                "절대 가격 추세는 양호하지만 benchmark 대비 상대강도는 약해지고 있습니다."
            )

        if flow["view"] == "NotAvailable":
            conflict_parts.append(
                "해당 자산은 수급 데이터가 없어 flow 기반 확인은 제한됩니다."
            )

        if relative_strength["view"] == "NotAvailable":
            conflict_parts.append(
                "상대강도 benchmark가 없어 시장 대비 우위 판단은 제한됩니다."
            )

        if not conflict_parts:
            conflict_parts.append("주요 판단 블록 간 큰 충돌은 관찰되지 않습니다.")

        return base + " ".join(conflict_parts)

    def generate_report_for_asset(self, row):
        trend = self.judge_trend(row)
        momentum = self.judge_momentum(row)
        flow = self.judge_flow(row)
        relative_strength = self.judge_relative_strength(row)
        risk = self.judge_risk(row)

        timing = self.judge_timing_readiness(
            row=row,
            trend=trend,
            momentum=momentum,
            flow=flow,
            relative_strength=relative_strength,
            risk=risk
        )

        stance, confidence, decision_basis = self.synthesize_stance(
            trend, momentum, flow, relative_strength, risk
        )

        evidence = []
        limitations = []
        risk_flags = []

        for block in [trend, momentum, flow, relative_strength, risk]:
            evidence.extend(block.get("evidence", []))
            limitations.extend(block.get("limitations", []))
            risk_flags.extend(block.get("risk_flags", []))

        evidence = evidence[:8]
        limitations = limitations[:6]
        risk_flags = risk_flags[:6]

        rationale = self.build_conflict_aware_rationale(
            row=row,
            stance=stance,
            confidence=confidence,
            decision_basis=decision_basis,
            trend=trend,
            momentum=momentum,
            flow=flow,
            relative_strength=relative_strength,
            risk=risk,
            timing=timing
        )

        return {
            "agent": "Market Agent",
            "as_of_date": str(row["Date"].date()),
            "target": {
                "asset_code": row["AssetCode"],
                "asset_name": row["AssetName"],
                "market": row["Market"]
            },
            "stance": stance,
            "confidence": confidence,
            "decision_basis": decision_basis,
            "technical_view": trend,
            "momentum_view": momentum,
            "flow_view": flow,
            "relative_strength_view": relative_strength,
            "risk_view": risk,
            "timing_readiness": timing,
            "evidence": evidence,
            "limitations": limitations,
            "risk_flags": risk_flags,
            "rationale": rationale
        }

    def build_reports(self, feature_df):
        latest_rows = (
            feature_df
            .sort_values(["AssetCode", "Date"])
            .groupby("AssetCode")
            .tail(1)
            .reset_index(drop=True)
        )

        reports = [self.generate_report_for_asset(row) for _, row in latest_rows.iterrows()]

        report_table = []

        for report in reports:
            report_table.append({
                "Date": report["as_of_date"],
                "AssetCode": report["target"]["asset_code"],
                "AssetName": report["target"]["asset_name"],
                "Market": report["target"]["market"],
                "Stance": report["stance"],
                "Confidence": report["confidence"],
                "TimingScore": report["timing_readiness"]["score"],
                "TimingZone": report["timing_readiness"]["zone"],
                "Trend": report["technical_view"]["view"],
                "Momentum": report["momentum_view"]["view"],
                "Flow": report["flow_view"]["view"],
                "RelativeStrength": report["relative_strength_view"]["view"],
                "Risk": report["risk_view"]["view"],
                "PositiveBlocks": ", ".join(report["decision_basis"]["positive_blocks"]),
                "NegativeBlocks": ", ".join(report["decision_basis"]["negative_blocks"]),
                "MixedBlocks": ", ".join(report["decision_basis"]["mixed_blocks"]),
                "Evidence": " | ".join(report["evidence"]),
                "Limitations": " | ".join(report["limitations"]),
                "RiskFlags": " | ".join(report["risk_flags"]),
                "TimingSupport": " | ".join(report["timing_readiness"]["supporting_signals"]),
                "TimingCaution": " | ".join(report["timing_readiness"]["caution_signals"]),
                "Rationale": report["rationale"]
            })

        report_table_df = pd.DataFrame(report_table)

        final_output = {
            "agent": "Market Agent",
            "report_type": "structured_market_analysis_with_timing",
            "generated_at": datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
            "n_assets": len(reports),
            "reports": reports
        }

        ensure_output_dir()
        with open(REPORT_JSON_FILE, "w", encoding="utf-8") as f:
            json.dump(final_output, f, ensure_ascii=False, indent=2)

        report_table_df.to_csv(REPORT_TABLE_FILE, index=False, encoding="utf-8-sig")

        print("[DONE] Rule-based reporting")
        print("json report:", REPORT_JSON_FILE)
        print("table report:", REPORT_TABLE_FILE)

        return final_output, report_table_df

    def build_txt_report(self, report_json):
        reports = report_json["reports"]
        generated_at = report_json["generated_at"]

        lines = []
        lines.append("=" * 70)
        lines.append("Market Agent Analysis Report")
        lines.append("=" * 70)
        lines.append(f"생성 시각: {generated_at}")
        lines.append(f"분석 자산 수: {len(reports)}")
        lines.append("")

        positive = sum(1 for r in reports if r["stance"] == "Positive")
        neutral = sum(1 for r in reports if r["stance"] == "Neutral")
        cautious = sum(1 for r in reports if r["stance"] == "Cautious")

        avg_timing = np.mean([r["timing_readiness"]["score"] for r in reports]) if reports else np.nan

        lines.append("[시장 전체 요약]")
        lines.append(f"- Positive: {positive}")
        lines.append(f"- Neutral : {neutral}")
        lines.append(f"- Cautious: {cautious}")
        lines.append(f"- 평균 Timing Readiness: {avg_timing:.1f}")

        if positive > cautious:
            overall = "전반적으로 우호 신호가 우세합니다."
        elif cautious > positive:
            overall = "전반적으로 경계 신호가 우세합니다."
        else:
            overall = "시장 방향성이 뚜렷하지 않은 중립 구간입니다."

        lines.append(f"- 종합 판단: {overall}")
        lines.append("")

        for r in reports:
            name = r["target"]["asset_name"]
            code = r["target"]["asset_code"]
            market = r["target"]["market"]

            lines.append("=" * 70)
            lines.append(f"{name} ({code}) - {market}")
            lines.append("=" * 70)
            lines.append(f"종합 판단: {r['stance']}")
            lines.append(f"신뢰도: {r['confidence']}")
            lines.append(f"Timing Readiness: {r['timing_readiness']['score']} / {r['timing_readiness']['zone']}")
            lines.append("")
            lines.append("[핵심 해석]")
            lines.append(r["rationale"])
            lines.append("")
            lines.append("[세부 관점]")
            lines.append(f"- Trend: {r['technical_view']['view']}")
            lines.append(f"- Momentum: {r['momentum_view']['view']}")
            lines.append(f"- Flow: {r['flow_view']['view']}")
            lines.append(f"- Relative Strength: {r['relative_strength_view']['view']}")
            lines.append(f"- Risk: {r['risk_view']['view']}")
            lines.append("")

            if r["evidence"]:
                lines.append("[근거]")
                for e in r["evidence"]:
                    lines.append(f"- {e}")
                lines.append("")

            if r["timing_readiness"]["supporting_signals"]:
                lines.append("[Timing Support]")
                for e in r["timing_readiness"]["supporting_signals"]:
                    lines.append(f"- {e}")
                lines.append("")

            if r["timing_readiness"]["caution_signals"]:
                lines.append("[Timing Caution]")
                for e in r["timing_readiness"]["caution_signals"]:
                    lines.append(f"- {e}")
                lines.append("")

            if r["risk_flags"]:
                lines.append("[리스크]")
                for rf in r["risk_flags"]:
                    lines.append(f"- {rf}")
                lines.append("")

            if r["limitations"]:
                lines.append("[해석 제한]")
                for lm in r["limitations"]:
                    lines.append(f"- {lm}")
                lines.append("")

        ensure_output_dir()
        with open(FINAL_TXT_FILE, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))

        print("[DONE] TXT report")
        print("txt report:", FINAL_TXT_FILE)

        return FINAL_TXT_FILE


# ============================================================
# [6] GPT Market Interpreter
# ============================================================

class GPTMarketInterpreter:
    def __init__(self, api_key=None, model="gpt-4.1-mini"):
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")

        if self.api_key is None or self.api_key.strip() == "":
            raise ValueError("OPENAI_API_KEY가 없습니다. 직접 입력하거나 환경변수로 설정하세요.")

        self.client = OpenAI(api_key=self.api_key)
        self.model = model

    def _make_compact_reports(self, report_json):
        compact_reports = []

        for r in report_json["reports"]:
            compact_reports.append({
                "asset_code": r["target"]["asset_code"],
                "asset_name": r["target"]["asset_name"],
                "market": r["target"]["market"],
                "as_of_date": r["as_of_date"],

                "rule_based_stance": r["stance"],
                "rule_based_confidence": r["confidence"],

                "rule_based_timing_readiness": r["timing_readiness"],

                "technical_view": r["technical_view"]["view"],
                "momentum_view": r["momentum_view"]["view"],
                "flow_view": r["flow_view"]["view"],
                "relative_strength_view": r["relative_strength_view"]["view"],
                "risk_view": r["risk_view"]["view"],

                "positive_blocks": r["decision_basis"]["positive_blocks"],
                "negative_blocks": r["decision_basis"]["negative_blocks"],
                "mixed_blocks": r["decision_basis"]["mixed_blocks"],

                "evidence": r["evidence"],
                "risk_flags": r["risk_flags"],
                "limitations": r["limitations"],
                "rule_based_rationale": r["rationale"]
            })

        return compact_reports

    def _make_prompt(self, report_json):
        compact_reports = self._make_compact_reports(report_json)

        schema = {
            "agent": "Market Agent",
            "decision_type": "llm_market_judgment",
            "overall_market_judgment": {
                "stance": "Positive | Neutral | Cautious",
                "confidence": "High | Medium | Low",
                "market_agent_score": "float between -1.0 and 1.0",
                "summary": "Market Agent의 전체 판단 요약"
            },
            "asset_judgments": [
                {
                    "asset_code": "...",
                    "asset_name": "...",
                    "market": "...",
                    "llm_stance": "Positive | Neutral | Cautious",
                    "llm_confidence": "High | Medium | Low",
                    "market_signal_score": "float between -1.0 and 1.0",
                    "timing_readiness": {
                        "score": "integer between 0 and 100",
                        "zone": "Strong Entry Zone | Entry Candidate | Neutral | Defensive | Avoid",
                        "reason": "보수적 타이밍 판단 근거"
                    },
                    "reasoning": "LLM이 직접 판단한 핵심 근거",
                    "supporting_signals": ["..."],
                    "conflicting_signals": ["..."],
                    "risk_flags": ["..."],
                    "limitations": ["..."]
                }
            ],
            "integration_payload": {
                "agent_name": "Market Agent",
                "signal": "Positive | Neutral | Cautious",
                "confidence": "High | Medium | Low",
                "score": "float between -1.0 and 1.0",
                "timing_readiness_score": "integer between 0 and 100",
                "timing_readiness_zone": "Strong Entry Zone | Entry Candidate | Neutral | Defensive | Avoid",
                "key_evidence": ["..."],
                "key_risks": ["..."],
                "limitations": ["..."],
                "handoff_message": "Integration Agent에 전달할 요약"
            }
        }

        prompt = """
너는 Multi AI Agent 기반 반도체 주가 예측 시스템의 Market Agent다.

[중요]
다른 Agent가 담당하는 영역은 판단하지 마라.
- 뉴스 분석 금지
- 산업/반도체 공급망 분석 금지
- 기업 펀더멘털 분석 금지
- 백테스트 성능 평가 금지
- 최종 투자 추천 금지

[너의 역할]
너는 오직 현재 입력된 시장/가격/수급/상대강도/risk feature만 보고 Market Agent 관점의 최종 판단을 직접 내려야 한다.

[판단 대상]
- 시장 환경이 해당 자산에 우호적인가
- 현재 가격 흐름이 위험한가
- trend, momentum, flow, relative strength, risk 신호가 서로 일관적인가
- Integration Agent에 넘길 Market Agent의 독립 판단은 무엇인가
- 보수적 Timing Readiness가 어느 정도인가
- 단, 최종 Buy/Sell/Hold 추천은 하지 않는다

[Timing Readiness 기준]
- 80~100: Strong Entry Zone. 시장 환경과 타이밍이 모두 우호적
- 60~79: Entry Candidate. 진입 후보이나 추가 확인 필요
- 40~59: Neutral. 방향성 혼재
- 20~39: Defensive. 리스크 축소 우선
- 0~19: Avoid. 신규 진입 회피

[입력 데이터]
아래 rule-based 결과는 참고 자료일 뿐이다.
LLM은 이 결과를 그대로 복사하지 말고, 충돌 여부와 근거 강도를 재판단해야 한다.

[사용자 질문]
2026년 5월 29일 삼성전자의 투자주체별 거래 비율을 기반으로
외국인 수급이 향후 7일, 14일, 30일 주가 방향에 어떤 시사점을 가지는지 평가하라.

단, Market Agent는 가격, 거래량, 수급, 기술적 지표, 상대강도, 리스크만 사용한다.
뉴스, 펀더멘털, 산업 분석은 제외한다.

""" + json.dumps(compact_reports, ensure_ascii=False, indent=2) + """

[출력 규칙]
반드시 JSON만 출력하라.
markdown code block을 쓰지 마라.
입력에 없는 외부 사실을 만들지 마라.
Buy/Sell/Hold 같은 최종 투자 의견을 내지 마라.

[출력 JSON schema]
""" + json.dumps(schema, ensure_ascii=False, indent=2) + """

[score 기준]
- +1.0에 가까울수록 시장 환경이 강하게 우호적
- 0에 가까울수록 중립 또는 혼재
- -1.0에 가까울수록 경계적
"""

        return prompt

    def _parse_json_safely(self, text):
        try:
            return json.loads(text)
        except Exception:
            match = re.search(r"\{.*\}", text, flags=re.DOTALL)
            if match:
                return json.loads(match.group(0))
            raise ValueError("GPT 응답을 JSON으로 파싱하지 못했습니다.")

    def generate_llm_report(self, report_json):
        prompt = self._make_prompt(report_json)

        response = self.client.responses.create(
            model=self.model,
            input=prompt,
            temperature=0.2
        )

        text = response.output_text.strip()

        try:
            parsed = self._parse_json_safely(text)
        except Exception as e:
            parsed = {
                "agent": "Market Agent",
                "decision_type": "llm_market_judgment",
                "parse_error": str(e),
                "raw_text": text
            }

        output = {
            "agent": "Market Agent",
            "model": self.model,
            "generated_at": datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
            "llm_judgment": parsed,
            "raw_text": text
        }

        ensure_output_dir()
        with open(LLM_REPORT_FILE, "w", encoding="utf-8") as f:
            f.write(text)

        with open(LLM_JSON_FILE, "w", encoding="utf-8") as f:
            json.dump(output, f, ensure_ascii=False, indent=2)

        if isinstance(parsed, dict) and "integration_payload" in parsed:
            with open(INTEGRATION_PAYLOAD_FILE, "w", encoding="utf-8") as f:
                json.dump(parsed["integration_payload"], f, ensure_ascii=False, indent=2)

        print("[DONE] LLM Market Judgment")
        print("llm txt :", LLM_REPORT_FILE)
        print("llm json:", LLM_JSON_FILE)
        print("integration payload:", INTEGRATION_PAYLOAD_FILE)

        return output


# ============================================================
# [7] MarketAgent
# ============================================================

class MarketAgent:
    def __init__(
        self,
        start_date=START_DATE,
        end_date=END_DATE,
        kr_stocks=KR_STOCKS,
        kr_indexes=KR_INDEXES,
        global_assets=GLOBAL_ASSETS,
        use_gpt=False,
        openai_api_key=None,
        gpt_model="gpt-4.1-mini"
    ):
        self.collector = MarketDataCollector(
            start_date=start_date,
            end_date=end_date,
            kr_stocks=kr_stocks,
            kr_indexes=kr_indexes,
            global_assets=global_assets
        )

        self.feature_builder = MarketFeatureBuilder()
        self.reporter = MarketReporter()

        self.use_gpt = use_gpt
        self.gpt_interpreter = None

        if use_gpt:
            self.gpt_interpreter = GPTMarketInterpreter(
                api_key=openai_api_key,
                model=gpt_model
            )

    def collect(self, max_pages=250):
        return self.collector.collect(max_pages=max_pages)

    def build_features(self, price_df=None, flow_df=None):
        return self.feature_builder.build(price_df=price_df, flow_df=flow_df)

    def generate_report(self, feature_df=None):
        if feature_df is None:
            feature_df = pd.read_csv(FEATURE_FILE)
            feature_df["Date"] = pd.to_datetime(feature_df["Date"], errors="coerce")

        report_json, report_table_df = self.reporter.build_reports(feature_df)
        txt_file = self.reporter.build_txt_report(report_json)

        llm_report = None

        if self.use_gpt:
            llm_report = self.gpt_interpreter.generate_llm_report(report_json)

        return {
            "json": report_json,
            "table": report_table_df,
            "txt_file": txt_file,
            "llm_report": llm_report
        }

    def run(self, max_pages=250, collect_data=True):
        ensure_output_dir()

        if collect_data:
            price_df, flow_df = self.collect(max_pages=max_pages)
        else:
            price_df = pd.read_csv(PRICE_FILE)
            flow_df = pd.read_csv(FLOW_FILE)

        feature_df = self.build_features(price_df=price_df, flow_df=flow_df)
        report = self.generate_report(feature_df=feature_df)

        return {
            "price_df": price_df,
            "flow_df": flow_df,
            "feature_df": feature_df,
            "report": report
        }


# ============================================================
# [8] 실행 코드
# ============================================================

# 먼저 GPT 없이 rule-based + timing만 확인하려면 USE_GPT=False
# GPT 판단까지 실행하려면 USE_GPT=True
USE_GPT = True

OPENAI_API_KEY = read_openai_key() if USE_GPT else None

agent = MarketAgent(
    use_gpt=USE_GPT,
    openai_api_key=OPENAI_API_KEY,
    gpt_model="gpt-4.1-mini"
)

result = agent.run(
    max_pages=250,
    collect_data=True
)

price_df = result["price_df"]
flow_df = result["flow_df"]
feature_df = result["feature_df"]
report_table_df = result["report"]["table"]

display(report_table_df)

if USE_GPT:
    llm_report = result["report"]["llm_report"]
    llm_judgment = llm_report["llm_judgment"]

    print(json.dumps(
        llm_judgment,
        ensure_ascii=False,
        indent=2
    ))
else:
    print("[INFO] GPT 비활성화 상태입니다. Rule-based report만 생성되었습니다.")


[INFO] Current working directory: d:\University\4-1\5Capstone1\AIagent\practice\market_agent\agentic
[INFO] OpenAI key file path: D:\University\4-1\5Capstone1\AIagent\practice\market_agent\OpenAI_key.txt
[START] Data collection
[OK] FDR stock: 005930, rows=1576
[OK] FDR stock: 000660, rows=1576
[OK] FDR stock: 042700, rows=1576
[OK] FDR index: KS11, rows=1576
[OK] FDR index: KQ11, rows=1576
[OK] yfinance: SOXX, rows=1615
[OK] yfinance: SMH, rows=1615
[OK] yfinance: ^IXIC, rows=1615
[OK] yfinance: ^GSPC, rows=1615
[OK] yfinance: KRW=X, rows=1674
[OK] Naver flow: 005930, rows=3152
[OK] Naver flow: 000660, rows=3152
[OK] Naver flow: 042700, rows=3152
[DONE] Data collection
price: market_agent_data\market_price_data.csv (16014, 11)
flow : market_agent_data\market_flow_data.csv (9456, 7)


C:\Users\User\AppData\Local\Temp\ipykernel_11384\1929032376.py:665: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  price_df
C:\Users\User\AppData\Local\Temp\ipykernel_11384\1929032376.py:699: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flow_wide
C:\Users\User\AppData\Local\Temp\ipykernel_11384\1929032376.py:620: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be rem

[DONE] Feature building
feature: market_agent_data\market_feature_data.csv (16014, 56)
[DONE] Rule-based reporting
json report: market_agent_data\market_agent_report.json
table report: market_agent_data\market_agent_report_table.csv
[DONE] TXT report
txt report: market_agent_data\market_agent_final_report.txt
[DONE] LLM Market Judgment
llm txt : market_agent_data\market_agent_llm_report.txt
llm json: market_agent_data\market_agent_llm_report.json
integration payload: market_agent_data\market_agent_integration_payload.json


,Date,AssetCode,AssetName,Market,Stance,Confidence,TimingScore,TimingZone,Trend,Momentum,...,Risk,PositiveBlocks,NegativeBlocks,MixedBlocks,Evidence,Limitations,RiskFlags,TimingSupport,TimingCaution,Rationale
0,2026-06-05,000660,SK Hynix,KR_STOCK,Neutral,Low,56,Neutral,Bullish,Mixed,...,Moderate,"trend, relative_strength",flow,"momentum, risk",MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,,20일 변동성이 5.69%로 높습니다.,이동평균 정배열 | RSI가 안정적 상승 구간 | ROC60 양수 | 기관 20일 ...,MACD가 signal line 하회 | MACD histogram 음수 | 외국인...,"SK Hynix에 대한 Market Agent의 판단은 Neutral이며, 신뢰도는..."
1,2026-06-05,005930,Samsung Electronics,KR_STOCK,Positive,Medium,81,Strong Entry Zone,Bullish,Positive,...,Moderate,"trend, momentum, relative_strength",flow,risk,MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,,20일 변동성이 4.74%로 높습니다.,이동평균 정배열 | MACD가 signal line 상회 | MACD histogr...,외국인+기관 20일 합산 수급 음수 | 외국인 20일 순매도 | 20일 변동성 높음...,Samsung Electronics에 대한 Market Agent의 판단은 Posi...
2,2026-06-05,042700,Hanmi Semiconductor,KR_STOCK,Cautious,High,0,Avoid,MildBearish,Negative,...,High,,"trend, momentum, flow, relative_strength, risk",,MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,,RSI14가 29.76로 과매도 구간입니다. | 52주 고점 대비 낙폭이 -30.8...,이동평균 정배열,MACD가 signal line 하회 | MACD histogram 음수 | RSI...,Hanmi Semiconductor에 대한 Market Agent의 판단은 Caut...
3,2026-06-05,KQ11,KOSDAQ,KR_INDEX,Neutral,Medium,19,Avoid,MildBearish,Negative,...,Moderate,,"trend, momentum",risk,MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,해당 자산에는 외국인/기관 수급 데이터가 없습니다. | 비교 가능한 상대강도 ben...,RSI14가 29.89로 과매도 구간입니다. | 52주 고점 대비 낙폭이 -18.2...,이동평균 정배열,MACD가 signal line 하회 | MACD histogram 음수 | RSI...,"KOSDAQ에 대한 Market Agent의 판단은 Neutral이며, 신뢰도는 M..."
4,2026-06-06,KRW=X,USD/KRW,FX,Neutral,Medium,84,Strong Entry Zone,Bullish,Positive,...,Moderate,"trend, momentum",,risk,MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,해당 자산에는 외국인/기관 수급 데이터가 없습니다. | 비교 가능한 상대강도 ben...,RSI14가 77.33로 과열 구간입니다. | RSI14가 77.33로 과열 리스크...,이동평균 정배열 | MACD가 signal line 상회 | MACD histogr...,RSI 단기 과열 접근 | 수급 데이터 부재 | 상대강도 benchmark 부재 |...,"USD/KRW에 대한 Market Agent의 판단은 Neutral이며, 신뢰도는 ..."
5,2026-06-05,KS11,KOSPI,KR_INDEX,Neutral,High,90,Strong Entry Zone,Bullish,Positive,...,Low,"trend, momentum",,,MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,해당 자산에는 외국인/기관 수급 데이터가 없습니다. | 비교 가능한 상대강도 ben...,20일 변동성이 3.42%로 중간 수준입니다.,이동평균 정배열 | MACD가 signal line 상회 | MACD histogr...,수급 데이터 부재 | 상대강도 benchmark 부재 | 20일 변동성 중간 수준,"KOSPI에 대한 Market Agent의 판단은 Neutral이며, 신뢰도는 Hi..."
6,2026-06-05,SMH,VanEck Semiconductor ETF,US_ETF,Neutral,Low,69,Entry Candidate,Bullish,Mixed,...,Low,trend,,"momentum, relative_strength",MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,해당 자산에는 외국인/기관 수급 데이터가 없습니다.,20일 변동성이 3.25%로 중간 수준입니다.,이동평균 정배열 | RSI가 안정적 상승 구간 | ROC60 양수 | 거래량이 평균 이상,MACD가 signal line 하회 | MACD histogram 음수 | 수급 ...,VanEck Semiconductor ETF에 대한 Market Agent의 판단은...
7,2026-06-05,SOXX,iShares Semiconductor ETF,US_ETF,Neutral,Medium,68,Entry Candidate,Bullish,Mixed,...,Moderate,"trend, relative_strength",,"momentum, risk",MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,해당 자산에는 외국인/기관 수급 데이터가 없습니다.,20일 변동성이 3.90%로 높습니다.,이동평균 정배열 | RSI가 안정적 상승 구간 | ROC60 양수 | benchma...,MACD가 signal line 하회 | MACD histogram 음수 | 수급 ...,iShares Semiconductor ETF에 대한 Market Agent의 판단...
8,2026-06-05,^GSPC,S&P 500,US_INDEX,Neutral,Medium,72,Entry Candidate,Bullish,Mixed,...,Low,trend,,momentum,MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,해당 자산에는 외국인/기관 수급 데이터가 없습니다. | 비교 가능한 상대강도 ben...,,이동평균 정배열 | RSI가 안정적 상승 구간 | ROC60 양수 | 20일 변동성 낮음,MACD가 signal line 하회 | MACD histogram 음수 | 수급 ...,"S&P 500에 대한 Market Agent의 판단은 Neutral이며, 신뢰도는 ..."
9,2026-06-05,^IXIC,NASDAQ Composite,US_INDEX,Neutral,Medium,66,Entry Candidate,Bullish,Mixed,...,Low,trend,,momentum,MA30 > MA60 > MA120 정배열이 형성되어 있습니다. | 현재 가격이 M...,RSI14가 41.29로 뚜렷한 방향성을 보이지 않습니다. | 해당 자산에는 외국인...,,이동평균 정배열 | ROC60 양수 | 거래량이 평균 이상 | 20일 변동성 낮음,MACD가 signal line 하회 | MACD histogram 음수 | 수급 ...,NASDAQ Composite에 대

{
  "agent": "Market Agent",
  "decision_type": "llm_market_judgment",
  "overall_market_judgment": {
    "stance": "Neutral",
    "confidence": "Medium",
    "market_agent_score": 0.3,
    "summary": "삼성전자 주가는 현재 강한 추세와 모멘텀, 상대강도 우위 신호를 보이나, 외국인 수급이 20일 기준 순매도 상태이고 기관과 외국인 합산 수급도 음수이며 변동성도 높아 수급 흐름은 약한 편입니다. 따라서 시장 환경은 우호적이지만 수급 불확실성으로 인해 단기 가격 방향성에는 주의가 필요합니다."
  },
  "asset_judgments": [
    {
      "asset_code": "005930",
      "asset_name": "Samsung Electronics",
      "market": "KR_STOCK",
      "llm_stance": "Neutral",
      "llm_confidence": "Medium",
      "market_signal_score": 0.35,
      "timing_readiness": {
        "score": 75,
        "zone": "Entry Candidate",
        "reason": "가격과 기술적 지표는 강한 상승 추세와 모멘텀을 나타내지만, 외국인 수급이 20일 순매도이며 외국인+기관 합산 수급도 음수이고 변동성도 높아 수급 흐름이 약한 점이 진입 타이밍에 대한 추가 확인 필요성을 제기합니다."
      },
      "reasoning": "이동평균선 정배열과 MACD, RSI 등 기술적 지표가 긍정적 신호를 주고 있으며 상대강도도 개선되어 추세와 모멘텀은 우호적입니다. 그러나 외국인 수급이 지속적으로 순매도 상태이고 기관과 외국인 합산 수급도 음수이며 변동성도 높아 단기 가격 흐름에 위험 요소가